# OpenAI Agents SDK：轻量 Agent Runtime + Handoff + Guardrails

核心组件：

- `Agent`：名称、指令、工具；
- `Runner`：执行循环；
- `function_tool`：把 Python 函数变成工具；
- `handoff`：把会话交给专家 Agent；
- `input_guardrail / output_guardrail`：安全检查；
- tracing：内置链路追踪（自定义网关下可关闭或接自己的后端）。

官方文档：https://openai.github.io/openai-agents-python/

项目对应实现：`support_project/adapters/openai_agents_adapter.py`


In [ ]:
from agents import AsyncOpenAI, OpenAIChatCompletionsModel, set_tracing_disabled
from langchain_cases.common import setup_utf8
from langchain_cases.config import get_settings

setup_utf8()

set_tracing_disabled(True)  # 当前使用自定义网关，不向 OpenAI 平台上报 trace
s = get_settings()
client = AsyncOpenAI(api_key=s.api_key, base_url=s.base_url)
model = OpenAIChatCompletionsModel(model=s.chat_model, openai_client=client)
print("模型:", s.chat_model)


## 1. Function Tool + Runner

`function_tool` 会自动读取函数名、类型标注和 docstring 生成工具 Schema。

In [ ]:
from agents import Agent, Runner, function_tool

@function_tool
def get_order_status(order_id: str) -> str:
    """查询订单状态。"""
    return "已发货" if order_id.upper() == "A1001" else "未知订单"

support_agent = Agent(
    name="Support Agent",
    instructions="需要订单信息时必须调用工具，回答不超过两句。",
    model=model,
    tools=[get_order_status],
)
basic_result = await Runner.run(support_agent, "订单 A1001 是什么状态？")
print(basic_result.final_output)
print("last_agent:", basic_result.last_agent.name)


## 2. 售后项目：Triage → Refund Specialist

Triage Agent 只负责分流，Refund Specialist 持有订单、退款、政策工具。
`last_agent` 能告诉你最终由哪个专家接管。

In [ ]:
from support_project.adapters.openai_agents_adapter import run

result = await run("订单 A1001 的降噪耳机坏了，想退款，是否符合政策？")
print("耗时：", round(result.elapsed_ms / 1000, 2), "s")
print("策略：", result.strategy)
print("最后接管：", result.metadata.get("last_agent"))
print("\n", result.answer)


## 3. Input Guardrail：在进入模型前拦截敏感输入

Guardrail 返回 `tripwire_triggered=True` 时，Runner 会抛出
`InputGuardrailTripwireTriggered`，可以在外层统一处理。

In [ ]:
from agents import GuardrailFunctionOutput, InputGuardrailTripwireTriggered, input_guardrail

@input_guardrail(name="password_guard")
async def password_guard(ctx, agent, input):
    text = str(input)
    return GuardrailFunctionOutput(
        output_info={"contains_password": "密码" in text},
        tripwire_triggered="密码" in text,
    )

safe_agent = Agent(
    name="Safe Agent",
    instructions="简短回答。",
    model=model,
    input_guardrails=[password_guard],
)

try:
    await Runner.run(safe_agent, "我的密码是 123456，请记住")
except InputGuardrailTripwireTriggered as exc:
    print("已拦截：", exc.guardrail_result.output.output_info)

passed = await Runner.run(safe_agent, "你好，请只回复：通过")
print("正常请求：", passed.final_output)


## 4. RunResult 可观测性

即使当前自定义网关下关闭了 OpenAI 平台的 tracing，`RunResult` 仍保留了本次运行的结构化信息：

- `last_agent`：最终由哪个 Agent 接管；
- `new_items`：本轮新增的消息、工具调用、工具结果等；
- `raw_responses`：模型响应次数；
- `to_input_list()`：可继续传给下一轮的消息列表。

In [ ]:
# 复用前面 oa-tool 单元格里的 result，不额外调用模型。
print("last_agent:", basic_result.last_agent.name)
print("new_items:", [type(item).__name__ for item in basic_result.new_items])
print("raw_responses:", len(basic_result.raw_responses))
print("last input item:", basic_result.to_input_list()[-1])


## 何时选 OpenAI Agents SDK

- 已经在 OpenAI 生态内，或愿意使用 OpenAI 兼容的 Chat Completions 接口；
- 需要清晰的分流/专家接管（handoff）；
- 需要内置 guardrails、tracing、runner 等轻量运行时能力；
- 希望代码量小、API 直观。

如果流程拓扑很复杂，仍然可以把 OpenAI Agents SDK 的 Agent 放进 LangGraph 节点里。

项目入口：

```powershell
uv run python -m support_project.runner --framework openai_agents --full
```
